<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/tiny-transformer/blob/main/Mini_Transformer_Text_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Prepare the data

In [ ]:
sentences = ["the cat sat on the mat the dog sat on the rug cats love milk dogs love bones the cat loves milk"]
special_tokens = ["<PAD>", "<START>", "<END>"]

tokens = word_tokenize(sentences[0].lower())
vocab_words = special_tokens + list(set(tokens))
vocab = {w:i for i,w in enumerate(vocab_words)}
id2word = {i:w for w,i in vocab.items()}
vocab_size = len(vocab)

In [ ]:
encoder_tokens = tokens
decoder_input_tokens = ["<START>"] + tokens
decoder_target_tokens = tokens + ["<END>"]

encoder_ids = tf.constant([vocab[w] for w in encoder_tokens], dtype=tf.int32)
decoder_input_ids = tf.constant([vocab[w] for w in decoder_input_tokens], dtype=tf.int32)
decoder_target_ids = tf.constant([vocab[w] for w in decoder_target_tokens], dtype=tf.int32)

Hyper parameters

In [ ]:
embedding_dim = 64
ffn_dim = 128
learning_rate = 0.001
epochs = 300

Layers

In [ ]:
embedding_layer = tf.keras.layers.Embedding(vocab_size, embedding_dim)
output_layer = tf.keras.layers.Dense(vocab_size)

layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
layernorm3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

ffn = tf.keras.Sequential([
    tf.keras.layers.Dense(ffn_dim, activation='relu'),
    tf.keras.layers.Dense(embedding_dim)
])

# Define Q/K/V Dense layers outside tf.function
Q_enc_layer = tf.keras.layers.Dense(embedding_dim)
K_enc_layer = tf.keras.layers.Dense(embedding_dim)
V_enc_layer = tf.keras.layers.Dense(embedding_dim)

Q_dec_layer = tf.keras.layers.Dense(embedding_dim)
K_dec_layer = tf.keras.layers.Dense(embedding_dim)
V_dec_layer = tf.keras.layers.Dense(embedding_dim)

Q_cross_layer = tf.keras.layers.Dense(embedding_dim)
K_cross_layer = tf.keras.layers.Dense(embedding_dim)
V_cross_layer = tf.keras.layers.Dense(embedding_dim)

optimizer = tf.keras.optimizers.Adam(learning_rate)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


Positional Encoding

In [ ]:
def positional_encoding_tf(seq_len, d_model):
    pos = tf.cast(tf.range(seq_len)[:, tf.newaxis], tf.float32)
    i = tf.cast(tf.range(d_model)[tf.newaxis, :], tf.float32)
    angles = pos / tf.pow(10000.0, (2 * (i//2)) / tf.cast(d_model, tf.float32))
    pe = tf.where(tf.math.floormod(i, 2) == 0, tf.sin(angles), tf.cos(angles))
    return pe

Attention

In [ ]:
def self_attention(x, Q_layer, K_layer, V_layer, mask=None):
    Q = Q_layer(x)
    K = K_layer(x)
    V = V_layer(x)
    scores = tf.matmul(Q, K, transpose_b=True) / tf.sqrt(tf.cast(embedding_dim, tf.float32))
    if mask is not None:
        scores += mask * -1e9
    weights = tf.nn.softmax(scores, axis=-1)
    return tf.matmul(weights, V)

def cross_attention(query, key_value):
    Q = Q_cross_layer(query)
    K = K_cross_layer(key_value)
    V = V_cross_layer(key_value)
    scores = tf.matmul(Q, K, transpose_b=True) / tf.sqrt(tf.cast(embedding_dim, tf.float32))
    weights = tf.nn.softmax(scores, axis=-1)
    return tf.matmul(weights, V)

Attention

In [ ]:

@tf.function
def train_step(enc_ids, dec_in_ids, dec_out_ids):
    with tf.GradientTape() as tape:
        # Encoder
        enc_emb = embedding_layer(enc_ids) + positional_encoding_tf(tf.shape(enc_ids)[0], embedding_dim)
        enc_out = layernorm1(enc_emb + self_attention(enc_emb, Q_enc_layer, K_enc_layer, V_enc_layer))

        # Decoder
        dec_emb = embedding_layer(dec_in_ids) + positional_encoding_tf(tf.shape(dec_in_ids)[0], embedding_dim)
        seq_len = tf.shape(dec_in_ids)[0]
        mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        dec_out = layernorm2(dec_emb + self_attention(dec_emb, Q_dec_layer, K_dec_layer, V_dec_layer, mask))

        # Cross attention
        dec_out = layernorm3(dec_out + cross_attention(dec_out, enc_out))

        # Feed-forward + logits
        dec_out = ffn(dec_out)
        logits = output_layer(dec_out)

        # Loss
        loss = loss_fn(dec_out_ids, logits)

    trainable_vars = (embedding_layer.trainable_variables + output_layer.trainable_variables +
                      ffn.trainable_variables +
                      Q_enc_layer.trainable_variables + K_enc_layer.trainable_variables + V_enc_layer.trainable_variables +
                      Q_dec_layer.trainable_variables + K_dec_layer.trainable_variables + V_dec_layer.trainable_variables +
                      Q_cross_layer.trainable_variables + K_cross_layer.trainable_variables + V_cross_layer.trainable_variables)

    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    return loss

Training Loop

In [ ]:
data = [(encoder_ids, decoder_input_ids, decoder_target_ids)]

for epoch in range(epochs):
    total_loss = 0
    for enc, din, dout in data:
        loss = train_step(enc, din, dout)
        total_loss += loss
    if epoch % 50 == 0:
        print(f"Epoch {epoch} Loss: {total_loss.numpy():.4f}")

Epoch 0 Loss: 3.9293
Epoch 50 Loss: 0.2487
Epoch 100 Loss: 0.0052
Epoch 150 Loss: 0.0020
Epoch 200 Loss: 0.0012
Epoch 250 Loss: 0.0008


In [ ]:
def generate_text(seed_sentence, max_len=10):
    enc = [vocab[w] for w in word_tokenize(seed_sentence.lower())]
    enc_ids = tf.constant(enc, dtype=tf.int32)
    enc_emb = embedding_layer(enc_ids) + positional_encoding_tf(tf.shape(enc_ids)[0], embedding_dim)
    enc_out = layernorm1(enc_emb + self_attention(enc_emb, Q_enc_layer, K_enc_layer, V_enc_layer))

    generated = [vocab["<START>"]]

    for _ in range(max_len):
        dec_ids = tf.constant(generated, dtype=tf.int32)
        dec_emb = embedding_layer(dec_ids) + positional_encoding_tf(tf.shape(dec_ids)[0], embedding_dim)
        seq_len = tf.shape(dec_ids)[0]
        mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        dec_out = layernorm2(dec_emb + self_attention(dec_emb, Q_dec_layer, K_dec_layer, V_dec_layer, mask))
        dec_out = layernorm3(dec_out + cross_attention(dec_out, enc_out))
        dec_out = ffn(dec_out)
        logits = output_layer(dec_out)
        next_id = tf.argmax(logits[-1]).numpy()
        if next_id == vocab["<END>"]:
            break
        generated.append(next_id)

    return " ".join([id2word[i] for i in generated[1:]])

In [ ]:
print(generate_text("the cat"))

sat sat sat on on on sat sat sat sat


MORE FINE TUNE

In [ ]:
import tensorflow as tf
import nltk
from nltk.tokenize import word_tokenize

# --------------------------------
# Setup
# --------------------------------
nltk.download("punkt")

# --------------------------------
# Data
# --------------------------------
raw_text = "dinesh loves reshma but reshma loves cat maha loves megalingam the cat sat on the mat the dog sat on the rug cats love milk dogs love bones the cat loves milk"
tokens = word_tokenize(raw_text.lower())

special_tokens = ["<PAD>", "<START>", "<END>"]
vocab_words = special_tokens + sorted(set(tokens))
vocab = {w: i for i, w in enumerate(vocab_words)}
id2word = {i: w for w, i in vocab.items()}
vocab_size = len(vocab)

PAD = vocab["<PAD>"]
START = vocab["<START>"]
END = vocab["<END>"]

seq_len = 8
enc_data, dec_in_data, dec_out_data = [], [], []

for i in range(len(tokens) - seq_len):
    enc = tokens[i:i+seq_len]
    dec_in = ["<START>"] + enc
    dec_out = enc + ["<END>"]

    enc_data.append([vocab[w] for w in enc])
    dec_in_data.append([vocab[w] for w in dec_in])
    dec_out_data.append([vocab[w] for w in dec_out])

enc_data = tf.constant(enc_data, tf.int32)
dec_in_data = tf.constant(dec_in_data, tf.int32)
dec_out_data = tf.constant(dec_out_data, tf.int32)

dataset = tf.data.Dataset.from_tensor_slices((enc_data, dec_in_data, dec_out_data))
dataset = dataset.shuffle(32).batch(4)

# --------------------------------
# Hyperparameters
# --------------------------------
embedding_dim = 48
ffn_dim = 96
epochs = 1200
learning_rate = 0.0003

# --------------------------------
# Layers
# --------------------------------
embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
output_layer = tf.keras.layers.Dense(vocab_size)
dropout = tf.keras.layers.Dropout(0.2)

ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
ln3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

ffn = tf.keras.Sequential([
    tf.keras.layers.Dense(ffn_dim, activation="relu"),
    tf.keras.layers.Dense(embedding_dim)
])

Qe = tf.keras.layers.Dense(embedding_dim)
Ke = tf.keras.layers.Dense(embedding_dim)
Ve = tf.keras.layers.Dense(embedding_dim)

Qd = tf.keras.layers.Dense(embedding_dim)
Kd = tf.keras.layers.Dense(embedding_dim)
Vd = tf.keras.layers.Dense(embedding_dim)

Qc = tf.keras.layers.Dense(embedding_dim)
Kc = tf.keras.layers.Dense(embedding_dim)
Vc = tf.keras.layers.Dense(embedding_dim)

optimizer = tf.keras.optimizers.Adam(learning_rate)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# --------------------------------
# Positional Encoding
# --------------------------------
def positional_encoding(L, D):
    pos = tf.cast(tf.range(L)[:, None], tf.float32)
    i = tf.cast(tf.range(D)[None, :], tf.float32)
    angle = pos / tf.pow(10000.0, (2 * tf.floor(i / 2)) / tf.cast(D, tf.float32))
    return tf.where(i % 2 == 0, tf.sin(angle), tf.cos(angle))

# --------------------------------
# Attention
# --------------------------------
def attention(q, k, v, mask=None):
    scores = tf.matmul(q, k, transpose_b=True)
    scores /= tf.sqrt(tf.cast(embedding_dim, tf.float32))
    if mask is not None:
        scores += mask * -1e9
    return tf.matmul(tf.nn.softmax(scores), v)

# --------------------------------
# Training Step
# --------------------------------
@tf.function
def train_step(enc, dec_in, dec_out):
    with tf.GradientTape() as tape:
        enc_emb = embedding(enc) + positional_encoding(seq_len, embedding_dim)
        enc_out = ln1(enc_emb + dropout(attention(Qe(enc_emb), Ke(enc_emb), Ve(enc_emb))))

        dec_emb = embedding(dec_in) + positional_encoding(seq_len + 1, embedding_dim)
        L = tf.shape(dec_in)[1]
        mask = 1 - tf.linalg.band_part(tf.ones((L, L)), -1, 0)

        dec_self = attention(Qd(dec_emb), Kd(dec_emb), Vd(dec_emb), mask)
        dec_out1 = ln2(dec_emb + dropout(dec_self))

        dec_cross = attention(Qc(dec_out1), Kc(enc_out), Vc(enc_out))
        dec_out2 = ln3(dec_out1 + dropout(dec_cross))

        logits = output_layer(ffn(dec_out2))
        loss = loss_fn(dec_out, logits)

    grads = tape.gradient(loss, tape.watched_variables())
    optimizer.apply_gradients(zip(grads, tape.watched_variables()))
    return loss

# --------------------------------
# Train
# --------------------------------
for epoch in range(epochs):
    total_loss = 0.0
    for enc, din, dout in dataset:
        total_loss += train_step(enc, din, dout)
    if epoch % 100 == 0:
        print(f"Epoch {epoch} Loss: {total_loss.numpy():.4f}")

# --------------------------------
# Generation (FIXED)
# --------------------------------
def generate(seed, max_len=8):
    seed_tokens = word_tokenize(seed.lower())[:seq_len]
    seed_ids = [vocab[w] for w in seed_tokens]
    seed_ids += [PAD] * (seq_len - len(seed_ids))

    enc = tf.constant([seed_ids], tf.int32)
    enc_emb = embedding(enc) + positional_encoding(seq_len, embedding_dim)
    enc_out = ln1(enc_emb + attention(Qe(enc_emb), Ke(enc_emb), Ve(enc_emb)))

    out = [START]
    for _ in range(max_len):
        dec_ids = out + [PAD] * (seq_len + 1 - len(out))
        dec = tf.constant([dec_ids], tf.int32)

        dec_emb = embedding(dec) + positional_encoding(seq_len + 1, embedding_dim)
        L = tf.shape(dec)[1]
        mask = 1 - tf.linalg.band_part(tf.ones((L, L)), -1, 0)

        d1 = ln2(dec_emb + attention(Qd(dec_emb), Kd(dec_emb), Vd(dec_emb), mask))
        d2 = ln3(d1 + attention(Qc(d1), Kc(enc_out), Vc(enc_out)))
        logits = output_layer(ffn(d2))

        next_id = tf.argmax(logits[0, len(out)-1]).numpy()
        if next_id == END:
            break
        out.append(next_id)

    return " ".join(id2word[i] for i in out[1:])

# --------------------------------
# Test
# --------------------------------
print(generate("the cat"))
print(generate("the dog"))
print(generate("dinesh"))
print(generate("reshma"))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Epoch 0 Loss: 19.2780
Epoch 100 Loss: 0.6010
Epoch 200 Loss: 0.0462
Epoch 300 Loss: 0.0121
Epoch 400 Loss: 0.0053
Epoch 500 Loss: 0.0028
Epoch 600 Loss: 0.0016
Epoch 700 Loss: 0.0010
Epoch 800 Loss: 0.0006
Epoch 900 Loss: 0.0004
Epoch 1000 Loss: 0.0003
Epoch 1100 Loss: 0.0002
the cat sat on the rug cats love
the mat the dog sat on the rug
dinesh loves megalingam the cat sat on the
reshma but reshma loves megalingam the cat sat


In [ ]:
import tensorflow as tf
import nltk
from nltk.tokenize import word_tokenize
import math

# --------------------------------
# Setup
# --------------------------------
nltk.download("punkt")

# --------------------------------
# Data
# --------------------------------
raw_text = "birds fly in the sky fish swim in the ocean cats chase mice dogs bark loudly cows eat grass sun rises in the east stars shine at night"
tokens = word_tokenize(raw_text.lower())

special_tokens = ["<PAD>", "<START>", "<END>"]
vocab_words = special_tokens + sorted(set(tokens))
vocab = {w: i for i, w in enumerate(vocab_words)}
id2word = {i: w for w, i in vocab.items()}
vocab_size = len(vocab)

PAD = vocab["<PAD>"]
START = vocab["<START>"]
END = vocab["<END>"]

seq_len = 8
enc_data, dec_in_data, dec_out_data = [], [], []

for i in range(len(tokens) - seq_len):
    enc = tokens[i:i+seq_len]
    dec_in = ["<START>"] + enc
    dec_out = enc + ["<END>"]

    enc_data.append([vocab[w] for w in enc])
    dec_in_data.append([vocab[w] for w in dec_in])
    dec_out_data.append([vocab[w] for w in dec_out])

enc_data = tf.constant(enc_data, tf.int32)
dec_in_data = tf.constant(dec_in_data, tf.int32)
dec_out_data = tf.constant(dec_out_data, tf.int32)

dataset = tf.data.Dataset.from_tensor_slices(
    (enc_data, dec_in_data, dec_out_data)
).shuffle(32).batch(4)

# --------------------------------
# Hyperparameters
# --------------------------------
embedding_dim = 48
ffn_dim = 96
epochs = 800
learning_rate = 3e-4

# --------------------------------
# Layers
# --------------------------------
embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
output_layer = tf.keras.layers.Dense(vocab_size)
dropout = tf.keras.layers.Dropout(0.2)

ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
ln3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

ffn = tf.keras.Sequential([
    tf.keras.layers.Dense(ffn_dim, activation="relu"),
    tf.keras.layers.Dense(embedding_dim)
])

Qe = tf.keras.layers.Dense(embedding_dim)
Ke = tf.keras.layers.Dense(embedding_dim)
Ve = tf.keras.layers.Dense(embedding_dim)

Qd = tf.keras.layers.Dense(embedding_dim)
Kd = tf.keras.layers.Dense(embedding_dim)
Vd = tf.keras.layers.Dense(embedding_dim)

Qc = tf.keras.layers.Dense(embedding_dim)
Kc = tf.keras.layers.Dense(embedding_dim)
Vc = tf.keras.layers.Dense(embedding_dim)

optimizer = tf.keras.optimizers.Adam(learning_rate)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# --------------------------------
# Positional Encoding
# --------------------------------
def positional_encoding(L, D):
    pos = tf.cast(tf.range(L)[:, None], tf.float32)
    i = tf.cast(tf.range(D)[None, :], tf.float32)
    angle = pos / tf.pow(10000.0, (2 * tf.floor(i / 2)) / D)
    return tf.where(i % 2 == 0, tf.sin(angle), tf.cos(angle))

# --------------------------------
# Masks (FIXED)
# --------------------------------
def padding_mask(seq):
    # (batch, seq_len) -> (batch, seq_len, seq_len)
    mask = tf.cast(tf.equal(seq, PAD), tf.float32)
    return mask[:, None, :]  # broadcast later

def causal_mask(L):
    return 1 - tf.linalg.band_part(tf.ones((L, L)), -1, 0)

# --------------------------------
# Attention (expects mask: batch x L x L)
# --------------------------------
def attention(q, k, v, mask=None):
    scores = tf.matmul(q, k, transpose_b=True)
    scores /= tf.sqrt(tf.cast(embedding_dim, tf.float32))

    if mask is not None:
        scores += mask * -1e9

    weights = tf.nn.softmax(scores)
    return tf.matmul(weights, v)

# --------------------------------
# Training Step
# --------------------------------
@tf.function
def train_step(enc, dec_in, dec_out):
    with tf.GradientTape() as tape:

        # ----- Encoder -----
        enc_emb = embedding(enc) + positional_encoding(seq_len, embedding_dim)
        enc_mask = padding_mask(enc)
        enc_attn = attention(Qe(enc_emb), Ke(enc_emb), Ve(enc_emb), enc_mask)
        enc_out = ln1(enc_emb + dropout(enc_attn))

        # ----- Decoder -----
        dec_emb = embedding(dec_in) + positional_encoding(seq_len + 1, embedding_dim)
        L = tf.shape(dec_in)[1]

        look_ahead = causal_mask(L)
        dec_self = attention(Qd(dec_emb), Kd(dec_emb), Vd(dec_emb), look_ahead)
        dec_out1 = ln2(dec_emb + dropout(dec_self))

        dec_cross = attention(Qc(dec_out1), Kc(enc_out), Vc(enc_out))
        dec_out2 = ln3(dec_out1 + dropout(dec_cross))

        logits = output_layer(ffn(dec_out2))
        loss = loss_fn(dec_out, logits)

    grads = tape.gradient(loss, tape.watched_variables())
    optimizer.apply_gradients(zip(grads, tape.watched_variables()))
    return loss

# --------------------------------
# Train
# --------------------------------
for epoch in range(epochs):
    total_loss = 0.0
    for enc, din, dout in dataset:
        total_loss += train_step(enc, din, dout)

    if epoch % 100 == 0:
        ppl = math.exp(total_loss.numpy())
        print(f"Epoch {epoch} | Loss: {total_loss:.4f} | PPL: {ppl:.2f}")

# --------------------------------
# Top-K Sampling
# --------------------------------
def sample_top_k(logits, k=5):
    values, indices = tf.math.top_k(logits, k)
    probs = tf.nn.softmax(values)
    choice = tf.random.categorical(tf.math.log([probs]), 1)
    return indices[choice[0, 0]]

# --------------------------------
# Generation
# --------------------------------
def generate(seed, max_len=8, k=5):
    seed_tokens = word_tokenize(seed.lower())[:seq_len]
    seed_ids = [vocab[w] for w in seed_tokens]
    seed_ids += [PAD] * (seq_len - len(seed_ids))

    enc = tf.constant([seed_ids])
    enc_emb = embedding(enc) + positional_encoding(seq_len, embedding_dim)
    enc_out = ln1(enc_emb + attention(Qe(enc_emb), Ke(enc_emb), Ve(enc_emb)))

    out = [START]
    for _ in range(max_len):
        dec_ids = out + [PAD] * (seq_len + 1 - len(out))
        dec = tf.constant([dec_ids])

        dec_emb = embedding(dec) + positional_encoding(seq_len + 1, embedding_dim)
        L = tf.shape(dec)[1]
        mask = causal_mask(L)

        d1 = ln2(dec_emb + attention(Qd(dec_emb), Kd(dec_emb), Vd(dec_emb), mask))
        d2 = ln3(d1 + attention(Qc(d1), Kc(enc_out), Vc(enc_out)))
        logits = output_layer(ffn(d2))

        next_id = sample_top_k(logits[0, len(out)-1], k).numpy()
        if next_id == END:
            break
        out.append(next_id)

    return " ".join(id2word[i] for i in out[1:])

# --------------------------------
# Test
# --------------------------------
print(generate("birds"))
print(generate("fish"))
print(generate("dogs"))
print(generate("sun"))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Epoch 0 | Loss: 17.2156 | PPL: 29966009.02
Epoch 100 | Loss: 0.3981 | PPL: 1.49
Epoch 200 | Loss: 0.0209 | PPL: 1.02
Epoch 300 | Loss: 0.0072 | PPL: 1.01
Epoch 400 | Loss: 0.0035 | PPL: 1.00
Epoch 500 | Loss: 0.0020 | PPL: 1.00
Epoch 600 | Loss: 0.0013 | PPL: 1.00
Epoch 700 | Loss: 0.0008 | PPL: 1.00
birds fly in the ocean cats chase mice
fish swim in the ocean cats chase mice
dogs bark loudly cows eat grass sun rises
sun rises in the east stars shine at
